# Prática 03/11 · Abstenção alta em MT: o primeiro classificador
**Aprendizagem Computacional · 4º TIA · Aula 02 · versão INICIAL**

**Pergunta:** a partir do perfil do eleitorado, dá para apontar quais municípios de Mato Grosso tiveram **abstenção alta** (25% ou mais) no 1º turno de 2026?

O fluxo é o mesmo de sempre (cap. 9): **problema → alvo (y) e features (X) → treino/teste → baseline → modelo → métricas → interpretação**.
A novidade de hoje é a **classificação**: o alvo é *sim/não*, e a acurácia sozinha engana (cap. 12).

| Parte | O que você faz | Apostila |
|---|---|---|
| 1–2 | Carregar e olhar os dados (pronto) | cap. 2 e 4 |
| 3 | **TODO 1:** montar `X` e `y` | cap. 2 e 5 |
| 4 | **TODO 2:** separar treino e teste | cap. 9 |
| 5 | **TODO 3:** baseline | cap. 9 |
| 6 | **TODO 4:** regressão logística dentro de um Pipeline | cap. 5 e 12 |
| 7 | Matriz de confusão (pronto) | cap. 12 |
| 8 | **TODO 5:** mudar o limiar de decisão | cap. 12 |
| 9 | Verificação final | — |

**Como usar:** procure os `# TODO`, complete e rode as células de cima para baixo. A última célula confere tudo e mostra ✅ ou ❌.
Não precisa instalar nada no Colab. O notebook não grava nenhum arquivo.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SEMENTE = 42      # o mesmo sorteio para todo mundo
LIMIAR_ALTA = 25  # abstenção alta = 25% ou mais (definição da prática)
pd.set_option('display.precision', 2)

## 1 · Carregar os dados (pronto)
Uma linha por município (142). Fonte: TSE, perfil do eleitorado 2026 e resultado do 1º turno de 04/10/2026, já juntados pelo professor (é a mesma base do projeto-molde da PPI).

A célula procura o CSV nesta ordem: pasta `dados/` ao lado do notebook (computador do laboratório) → GitHub da disciplina (Colab, precisa de internet).
Se as duas falharem, arraste `municipios_mt_abstencao.csv` para a aba **Arquivos** (ícone de pasta, à esquerda, no Colab) e rode de novo.

In [ ]:
ARQUIVO = 'municipios_mt_abstencao.csv'
URL = 'https://raw.githubusercontent.com/Schavess/ifmt-vg-ac/main/atividades/2026-11-03_Aula02_Abstencao_Alta/dados/municipios_mt_abstencao.csv'

candidatos = [Path('dados') / ARQUIVO, Path('..') / 'dados' / ARQUIVO, Path(ARQUIVO)]
local = next((p for p in candidatos if p.exists()), None)
if local is not None:
    municipios = pd.read_csv(local)
    print('Lido do arquivo:', local.resolve())
else:
    municipios = pd.read_csv(URL)
    print('Lido do GitHub da disciplina')

print(municipios.shape)
municipios.head()

## 2 · Olhar antes de modelar (pronto)
| Coluna | O que é |
|---|---|
| `CD_MUNICIPIO`, `NM_MUNICIPIO` | código e nome: **identificadores**, nunca features |
| `PCT_16_24` | % do eleitorado com 16 a 24 anos |
| `PCT_65_MAIS` | % com 65 anos ou mais (para quem o voto é facultativo a partir dos 70) |
| `PCT_SUPERIOR_COMPLETO` | % com ensino superior completo |
| `LOG_ELEITORADO` | logaritmo do tamanho do eleitorado (o log "encolhe" Cuiabá e Várzea Grande para perto dos outros) |
| `ELEITORADO_PERFIL`, `QT_APTOS` | tamanho do eleitorado (duas fontes) |
| `QT_ABSTENCOES`, `ABSTENCAO_PCT` | **o resultado da eleição**: quantos faltaram e o percentual |

In [ ]:
print(municipios['ABSTENCAO_PCT'].describe())
alta = municipios['ABSTENCAO_PCT'] >= LIMIAR_ALTA
print(f'\nMunicípios com abstenção alta: {alta.sum()} de {len(municipios)} ({alta.mean():.0%})')

municipios['ABSTENCAO_PCT'].plot.hist(bins=20, figsize=(6, 3), title='Abstenção nos 142 municípios de MT (%)')
plt.axvline(LIMIAR_ALTA, color='red', ls='--', label='limiar da prática (25%)')
plt.legend(); plt.show()

Só cerca de 3 em cada 10 municípios estão na classe "alta". Guarde esse número: ele vai explicar o resultado do baseline.

## 3 · TODO 1: montar X e y (cap. 2 e 5)
- `y` é **1** quando `ABSTENCAO_PCT >= LIMIAR_ALTA` e **0** caso contrário. Dica: `(condição).astype(int)`.
- `X` tem só as 4 colunas de perfil da lista `FEATURES`.

**Por que não usar as outras colunas?** `CD_MUNICIPIO` e o nome são só etiquetas. `QT_ABSTENCOES` e `ABSTENCAO_PCT` **são** a resposta: usar isso como feature é vazamento (o modelo "cola" o gabarito e não serviria para prever a próxima eleição).

In [ ]:
FEATURES = ['PCT_16_24', 'PCT_65_MAIS', 'PCT_SUPERIOR_COMPLETO', 'LOG_ELEITORADO']

# TODO 1a: y = 1 se ABSTENCAO_PCT >= LIMIAR_ALTA, senão 0
y = None
# TODO 1b: X = só as colunas de FEATURES
X = None

print(X.shape, '| positivos (abstenção alta):', y.sum())

## 4 · TODO 2: separar treino e teste ANTES de qualquer ajuste (cap. 9)
Use `train_test_split` com `test_size=0.25`, `random_state=SEMENTE` e `stratify=y`.

`stratify=y` garante a mesma proporção de municípios "alta" no treino e no teste. Com só 142 linhas, sem isso o teste poderia ficar com quase nenhum positivo.

In [ ]:
from sklearn.model_selection import train_test_split

# TODO 2: X_treino, X_teste, y_treino, y_teste = train_test_split(...)
X_treino, X_teste, y_treino, y_teste = None, None, None, None

print('treino:', X_treino.shape, '| teste:', X_teste.shape)
print(f'proporção de "alta" -> treino {y_treino.mean():.2f} | teste {y_teste.mean():.2f}')

## 5 · TODO 3: a régua mínima, o baseline (cap. 9)
O `DummyClassifier(strategy='most_frequent')` ignora o X e chuta sempre a classe mais comum do treino. Todo modelo precisa ganhar dele.

A função `avaliar` (pronta) imprime três métricas para a classe positiva, "alta":
- **acurácia:** % de acertos no total;
- **precisão:** dos que o modelo apontou como "alta", quantos eram mesmo;
- **recall:** dos que eram "alta", quantos o modelo encontrou.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix


def avaliar(nome, y_real, y_previsto):
    acc = accuracy_score(y_real, y_previsto)
    prec = precision_score(y_real, y_previsto, zero_division=0)
    rec = recall_score(y_real, y_previsto)
    print(f'{nome:<22} acurácia {acc:.2f} | precisão {prec:.2f} | recall {rec:.2f}')
    return acc, prec, rec


# TODO 3: crie o baseline, ajuste com fit(X_treino, y_treino) e preveja X_teste
baseline = None
prev_baseline = None

res_baseline = avaliar('Baseline (mais comum)', y_teste, prev_baseline)

**Pergunta para discussão:** o baseline acerta cerca de 7 em cada 10 municípios. Ele serve para a Secretaria que quer saber onde fazer campanha de mobilização?

**Resposta esperada:** não. Ele diz "baixa" para todo mundo, então o recall é 0: não encontra nenhum município com abstenção alta. A acurácia alta vem só de a classe "baixa" ser maioria. É por isso que, em classificação, a acurácia sozinha engana.

## 6 · TODO 4: regressão logística dentro de um Pipeline (cap. 5 e 12)
A **regressão logística** calcula uma probabilidade entre 0 e 1 de o município ser "alta" e, por padrão, aponta "alta" quando essa probabilidade passa de 0,5.

Monte `make_pipeline(StandardScaler(), LogisticRegression())`. O `StandardScaler` coloca as 4 colunas na mesma escala; dentro do Pipeline, ele aprende média e desvio **só do treino** (anti-vazamento, cap. 5).

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# TODO 4: crie o pipeline, ajuste no treino e preveja o teste
modelo = None
prev_modelo = None

res_baseline = avaliar('Baseline (mais comum)', y_teste, prev_baseline)
res_modelo = avaliar('Regressão logística', y_teste, prev_modelo)

## 7 · Matriz de confusão (pronto, cap. 12)
Linhas = o que aconteceu; colunas = o que o modelo disse. A diagonal são os acertos.
- **Falso negativo (FN):** era "alta" e o modelo disse "baixa" → município que **ficou sem campanha**.
- **Falso positivo (FP):** era "baixa" e o modelo disse "alta" → campanha **gasta à toa**.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, eixos = plt.subplots(1, 2, figsize=(9, 3.5))
for ax, prev, titulo in zip(eixos, [prev_baseline, prev_modelo], ['Baseline', 'Regressão logística']):
    ConfusionMatrixDisplay.from_predictions(y_teste, prev, display_labels=['baixa', 'alta'],
                                            ax=ax, colorbar=False)
    ax.set_title(titulo); ax.set_xlabel('O modelo disse'); ax.set_ylabel('Aconteceu')
plt.tight_layout(); plt.show()

vn, fp, fn, vp = confusion_matrix(y_teste, prev_modelo).ravel()
print(f'Logística: {vp} "alta" encontrados, {fn} perdidos (FN), {fp} alarmes falsos (FP)')

**O que o modelo aprendeu?** Com dados padronizados, o sinal do coeficiente diz a direção: positivo puxa para "alta", negativo puxa para "baixa".

In [ ]:
coeficientes = pd.Series(modelo[-1].coef_[0], index=FEATURES).sort_values()
print(coeficientes.round(2))

**Pergunta para discussão:** `PCT_SUPERIOR_COMPLETO` tem coeficiente negativo. Dá para dizer que "mais diploma faz o eleitor votar"?

**Resposta esperada:** não. É associação, não causa (correlação não é causalidade). Municípios com mais gente de nível superior costumam ser mais urbanos e ter outras diferenças (renda, distância das seções, cadastro atualizado) que também mexem na abstenção. O modelo só registra o padrão.

## 8 · TODO 5: mudar o limiar de decisão (cap. 12)
Perder um município "alta" (FN) é pior para a campanha do que mandar a equipe a um município que nem precisava (FP)? Se sim, podemos **baixar o limiar**: apontar "alta" a partir de 0,3 de probabilidade, e não de 0,5.

`modelo.predict_proba(X_teste)[:, 1]` dá a probabilidade de "alta" para cada município do teste.

In [ ]:
LIMIAR_DECISAO = 0.3

# TODO 5a: probabilidade de "alta" para cada município do teste
prob_alta = None
# TODO 5b: 1 se prob_alta >= LIMIAR_DECISAO, senão 0 (use .astype(int))
prev_limiar = None

res_modelo = avaliar('Logística (limiar 0,5)', y_teste, prev_modelo)
res_limiar = avaliar('Logística (limiar 0,3)', y_teste, prev_limiar)

**Pergunta para discussão:** o que ganhamos e o que perdemos ao baixar o limiar?

**Resposta esperada:** o recall sobe (encontramos mais municípios "alta") e a precisão cai (mais alarmes falsos). A escolha do limiar não é técnica: depende do custo de cada erro para quem decide. No projeto de vocês, escrevam esse raciocínio no artigo.

**Atenção:** aqui escolhemos o limiar olhando o teste só para aprender o efeito. No projeto, o limiar se escolhe com validação cruzada no treino (cap. 9 e 18), e o teste é aberto uma única vez.

## 9 · Verificação final
Rode sem alterar. Cada linha mostra ✅ ou ❌. Os valores esperados estão no `LEIAME.md` da prática.

In [ ]:
def conferir(descricao, teste):
    try:
        passou = bool(teste())
    except Exception:          # variável ainda vazia (None) ou célula não rodada
        passou = False
    print('✅' if passou else '❌', descricao)
    return passou

ok = [
    conferir('X tem 142 linhas e as 4 colunas de perfil', lambda: list(X.columns) == FEATURES and len(X) == 142),
    conferir('y tem 44 municípios com abstenção alta', lambda: int(y.sum()) == 44),
    conferir('teste tem 36 municípios, 11 deles "alta"', lambda: len(y_teste) == 36 and int(y_teste.sum()) == 11),
    conferir('baseline: acurácia 0,69 e recall 0,00', lambda: round(accuracy_score(y_teste, prev_baseline), 2) == 0.69 and recall_score(y_teste, prev_baseline) == 0),
    conferir('logística (0,5): acurácia 0,75 e recall 0,45', lambda: round(accuracy_score(y_teste, prev_modelo), 2) == 0.75 and round(recall_score(y_teste, prev_modelo), 2) == 0.45),
    conferir('logística (0,3): recall 0,73 e precisão 0,53', lambda: round(recall_score(y_teste, prev_limiar), 2) == 0.73 and round(precision_score(y_teste, prev_limiar), 2) == 0.53),
]
print(f'\n{sum(ok)} de {len(ok)} verificações OK')

## Checklist (as 6 perguntas da PPI)
1. **Qual é o y?** abstenção ≥ 25% (sim/não).
2. **Qual é o baseline?** chutar sempre "baixa".
3. **Por que essa métrica?** recall, porque o erro caro é deixar município "alta" sem campanha.
4. **Onde foi feito o split?** antes de qualquer ajuste, estratificado, com `random_state=42`.
5. **Há vazamento?** não: o scaler está no Pipeline e `QT_ABSTENCOES`/`ABSTENCAO_PCT` ficaram fora do X.
6. **A conclusão está apoiada em evidência?** só 36 municípios no teste: cada acerto a mais muda o recall em 9 pontos. A conclusão é "a logística supera o baseline", não "acerta 75% sempre".

**Para quem terminar:** troque `LogisticRegression()` por `KNeighborsClassifier(n_neighbors=5)` (de `sklearn.neighbors`) e compare. O que acontece se tirar o `StandardScaler`? (cap. 6 e 13)